## Aurora_V4 - kWh

#### <span style="color:red; font-weight:bold"> Instructions for Using this Jupyter Notebook:</span>
This Notebook is using end point value minus start point value of kWh to calculate the kWh within one fiscal year.
  
<span style="color:royalblue">(1) Place **data_clean.py** and this Notebook in the same folder (already done)</span>

<span style="color:royalblue">(2) Place **aurora_v4.meter_info.csv,** **special_meters.xlsx** and the **raw data files** in the same directory (already done)</span> 
- **aurora_v4.meter_info.csv** is exported from the database
- **NOTE:** Filename of the raw data files should include the variable name (i.e., kwh)

<span style="color:royalblue">(3) Modify the **Parameters in Section 1** as needed before running this Notebook:</span> 
- Time range, FY, etc.

<span style="color:royalblue">(4) Run the Notebook:</span> 
- Set **Checked = False** and run it once.
- Check **kwh_original_all.pdf** and **kwh_corrected_all.pdf** located in **output_dir**.
- If there are any unusual periods in the meter reading data, record them in **special_meters.xlsx** located in **input_dir**.
- If no problem, set **Checked = True** and run the notebook again.

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

Checked = True#False#   # Set to True after you checked the bad_meters plot

# Time Range: All Four Fiscal Year
start_time = "2021-07-01 00:00:00"
end_time = "2025-07-01 00:00:00"
FY = ".fy22_fy25"

######################################################

In [ ]:

# Data Directories
input_dir = "../data/extracts/"  # directory for raw data files & other input files

output_dir = "../data/outputs/"  # directory for data outputs (different from input_dir)

plot_dir = "../data/outputs/plots/"  # directory for plot outputs


# Variable
var = 'kwh'


# Input Files
var_file = input_dir + "aurora_v4." + var + FY + ".csv"  # data file 0
var_rejects_file = input_dir + "aurora_v4." + var + "_rejects" + FY + ".csv"  # data file 1

meter_info_file = input_dir + "aurora_v4.meter_info.csv"  # contains all meter information
special_meters_file = input_dir + "special_meters.xlsx"  # records special meters that need to be corrected


# # Output Files
# meter_annual_csv = output_dir + "meter_annual_" + var + FY + ".csv"  # annual kwh usage for each meter
# building_annual_csv = output_dir + "building_annual_" + var + FY + ".csv"  # annual kwh usage for each building


# Exclude: Meters of buildings equipped with PV and Student Health
meters_with_pv = ['bachman_hall_main', 'campus_ctr_main', 'dance_bldg_main', 'gartley_hall_main', 'warrior_rec_ctr_main']
meters_excluded = meters_with_pv + ['student_health_main']  # student_health data is in vitality_v5


# Data Frequency
freq = '15min'

# Schema
schema = 'aurora_v4'


### 2. Imports

In [3]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from IPython.display import display, Markdown
import math

import data_clean as dc  # import self-defined module


### 3. Load Data

In [4]:
# Read var and var_rejects tables
df0 = pd.read_csv(var_file, low_memory=False)
df1 = pd.read_csv(var_rejects_file, low_memory=False)


### 4. Data Processing

In [5]:
# The meter_reading values of these two meters in the kwh_rejects table are switched, so switch them back
swap_map = {
    "hale_aloha_ilima_tower_cafe": "hale_aloha_ilima_tower_main",
    "hale_aloha_ilima_tower_main": "hale_aloha_ilima_tower_cafe"
}

df1['meter_name'] = df1['meter_name'].replace(swap_map)


In [6]:
# Concatenate df0 and df1 vertically
combined_df = pd.concat([df0, df1], ignore_index=True)

# Convert 'datetime' to datetime type (if not already)
combined_df['datetime'] = pd.to_datetime(combined_df['datetime'])

# Sort by meter_name and datetime
combined_df = combined_df.sort_values(by=['meter_name', 'datetime']).reset_index(drop=True)


In [7]:
# Pivot table with every meter be one column
pivoted_df = combined_df.pivot(index='datetime', columns='meter_name', values='meter_reading').reset_index()

# Fill missing timestamps
full_df = dc.fill_missing_timestamps(pivoted_df, freq)


##### <span style="color:royalblue">Filter Meters and Time Range:</span>

In [8]:
### Filter One: Retain only main meters and filter out sub and PV meters ###

# Step 1: Read meter info
meter_info = pd.read_csv(meter_info_file)

# Step 2: Exclude certain meters first
all_meters = meter_info['meter_name'].unique()
non_exc_meters = [m for m in all_meters if m not in meters_excluded]

# Step 3: Get all 'main' meters from non-PV meters
main_meters_no_exc = meter_info[
    (meter_info['end_use'] == 'main') & 
    (meter_info['meter_name'].isin(non_exc_meters))]['meter_name'].unique()

# Step 4: Filter full_df to keep only non-PV 'main' meters (plus datetime)
columns_to_keep = ['datetime'] + list(full_df.columns.intersection(main_meters_no_exc))
filtered_df = full_df[columns_to_keep]

# Set index as datetime
filtered_df.set_index('datetime', inplace=True)


In [9]:
### Filter Two: Time Range ###

df = filtered_df.loc[start_time:end_time, :].copy()
df.index = pd.to_datetime(df.index)

# Initial Data Cleaning: Replace all 0s with NaN in the entire DataFrame 
df = df.replace(0, np.nan)

df.head(2)


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_science_main_1,ag_science_main_2,andrews_amp_main,archtecture_main,bachman_hall_annex,biomedical_science_main_a,biomedical_science_main_b,...,sinclair_lib_main,softball_tennis_main,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2021-07-01 00:00:00,480271.0,105489.0,1389205.0,2996452.00,1620210.00,23613.00,1768622.0,NaN,52164571.00,4434250.0,...,1877654.00,52933.00,5349896.00,22160240.0,22220366.00,12340282.00,73180.0,212538.00,3097509.0,194123.0
2021-07-01 00:15:00,480275.5,105489.5,1389208.0,2996495.25,1620232.75,23613.25,1768630.0,NaN,52164732.75,4434261.5,...,1877674.25,52935.25,5349930.25,22160300.5,22220391.75,12340289.25,73180.5,212539.75,3097515.0,194126.0


##### <span style="color:royalblue">Correct Special Meters:</span>
- See **special_meters.xlsx** in input_dir for details.

In [10]:
df_corrected = dc.apply_special_meter_corrections(df, special_meters_file)

##### <span style="color:royalblue">Plot original and corrected lines:</span>

In [11]:
dc.plot_data(df,var, "original", plot_dir)

✅ Multi-page PDF saved to: ../data/outputs/plots/kwh_original_all.pdf


In [12]:
dc.plot_data(df_corrected,var, "corrected", plot_dir)

✅ Multi-page PDF saved to: ../data/outputs/plots/kwh_corrected_all.pdf


In [13]:
# df.loc["2022-12-21":"2023-1-20", "ag_science_main_2"].plot()

In [14]:
# df["ag_science_main_2"].to_csv("TEST.csv", index=True)

In [15]:
df_bad_meters, df_restarts = dc.find_bad_meters(df_corrected)
df_restarts

,meter_name,previous_valid_time,restart_time
0,admin_serv_1,2022-01-21 12:30:00,2022-01-21 12:45:00
1,ag_science_main_1,2024-03-06 14:30:00,2024-03-06 15:45:00
2,andrews_amp_main,2023-11-15 11:45:00,2023-11-15 12:45:00
3,andrews_amp_main,2024-01-08 21:00:00,2024-01-08 22:00:00
4,andrews_amp_main,2024-01-26 07:30:00,2024-01-26 18:45:00
...,...,...,...
69,paradise_palms_main,2025-02-17 12:00:00,2025-02-17 12:45:00
70,parking_struct_ph_ii_main,2024-06-10 11:45:00,2024-06-17 10:30:00
71,pbrc_main_a,2024-06-10 12:00:00,2024-06-14 14:30:00
72,pbrc_main_a,2025-05-24 20:30:00,2025-05-24 20:45:00


In [16]:
if not Checked:
    raise SystemExit("Execution stopped because Checked = False")
    

### 5. Interpolation

In [17]:
# Copy data into a new dataframe
data = df_corrected

# Prepare flags DataFrame
interpolation_flags = pd.DataFrame(
    0,
    index=data.index,
    columns=[f'{col}_interpolated' for col in data.columns]
)

# Apply cleaning + interpolation to first 20 meters
for col in data.columns:
    data[col] = dc.clean_and_interpolate(
        column=data[col],
        flags=interpolation_flags[f'{col}_interpolated'],
        df_restarts=df_restarts,
        special_meters_file=special_meters_file
    )

# Replace all 0s with NaN in the entire DataFrame 
data = data.replace(0, np.nan)

In [18]:
dc.plot_data(data,var, "interpolated", plot_dir)

✅ Multi-page PDF saved to: ../data/outputs/plots/kwh_interpolated_all.pdf


##### <span style="color:royalblue">Combine data with interpolation_flags and Reshape the dataframe:</span>

In [19]:
final_df = dc.reshape_interpolated_data(data, interpolation_flags)

# Replace 0 value with NaN missing value
final_df['meter_reading'] = final_df['meter_reading'].replace(0, np.nan)

# Add an 'is_missing' column: 1 for missing values, 0 otherwise
final_df['is_missing'] = final_df['meter_reading'].isna().astype(int)

display(Markdown(f"<span style='color:royalblue; font-weight:bold'>Table of Interpolated {var}:</span>"))
final_df.head()

<span style='color:royalblue; font-weight:bold'>Table of Interpolated kwh:</span>

,datetime,meter_name,meter_reading,is_interpolated,is_missing
0,2021-07-01,admin_serv_1,480271.0,0,0
1,2021-07-01,admin_serv_2_main,105489.0,0,0
2,2021-07-01,ag_engineering_main,1389205.0,0,0
3,2021-07-01,ag_science_main_1,2996452.0,0,0
4,2021-07-01,ag_science_main_2,1620210.0,0,0


In [20]:
# Create the file name with the variable
file_name_i = f"{var}_{schema}_interpolated.csv"
# Save file
dc.save_file(final_df, file_name_i, output_dir)  # for uploading

Writing file: "../data/outputs/kwh_aurora_v4_interpolated.csv"


### 6. Interval

In [21]:
# Calculate interval
delta_df = dc.calculate_delta_df(data)

In [22]:
# Plot interval
dc.plot_data(delta_df,var, "interval", plot_dir)

✅ Multi-page PDF saved to: ../data/outputs/plots/kwh_interval_all.pdf


In [23]:
# Reshare dataframe for uplaoding
reshaped_delta_df = dc.reshape_delta_df(delta_df.reset_index(), var)
reshaped_delta_df.head()

,datetime,meter_name,delta_kwh
0,2021-07-01,admin_serv_1,NaN
1,2021-07-01,admin_serv_2_main,NaN
2,2021-07-01,ag_engineering_main,NaN
3,2021-07-01,ag_science_main_1,NaN
4,2021-07-01,ag_science_main_2,NaN


In [24]:
# Create the file name with the variable
file_name_iv = f"{var}_{schema}_interval.csv"
# Save file
dc.save_file(delta_df.reset_index(), 'delta_'+var+".csv", output_dir)  # for calculating energy_disagg
dc.save_file(reshaped_delta_df, file_name_iv, output_dir)  # for uploading

Writing file: "../data/outputs/delta_kwh.csv"
Writing file: "../data/outputs/kwh_aurora_v4_interval.csv"
